# 2. Caption pilot H — baseline + UPT hidden-pair memory
Một nhánh trên **HAI T4 bằng DDP**. Internet ON; Save & Run All.
Chạy sau khi cache notebook hoàn tất và đã lưu toàn bộ folder thành dataset.
A/H có thể chạy song song trên hai tài khoản, mỗi notebook T4x2. Chia sẻ/mount đúng các private dataset.
Cùng baseline, train10k/val1000 đã dùng trước,1250updates,batch8,CE200 rồi CE+.05SCST(CIDErD+.5object_count_v2).
Không dùng adapter crop/phrase. Giữ prompt legacy của baseline; không thêm verb text UPT.

In [ ]:
print('UPT HIDDEN H: BOOTSTRAP START',flush=True)
from pathlib import Path
ARM='H'
CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/qformer-scst-object-w05-legacy/qformer_scst_object_w05_legacy.pth')
LEGACY_PROMPT=Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
DETECTIONS=Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
HIDDEN_CACHE=Path('/kaggle/input/datasets/ducanh2403/upt-hidden-pairs-caption11k/upt_hidden_pairs_11k')
OUTPUT=Path('/kaggle/working/upt_hidden_caption_H')
# HIDDEN_CACHE là ví dụ tên dataset; sửa đúng thư mục chứa cache_receipt.json của bạn.
for path in (CHECKPOINT,LEGACY_PROMPT,COCO_JSON,DETECTIONS,HIDDEN_CACHE/'cache_receipt.json'):
    assert path.is_file(), f'Missing exact input: {path}'
assert VISUAL_CACHE.is_dir(),VISUAL_CACHE
assert not OUTPUT.exists(),'Use fresh output/session; preserve checkpoints.'


In [ ]:
import base64, hashlib, json, os, shutil, subprocess, sys, tempfile, zlib
os.environ.update(PYTHONUNBUFFERED='1', HDF5_USE_FILE_LOCKING='FALSE', PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_XET='1')
# Isolate each bootstrap: embedded tracked overlays from a prior run are intentional edits.
REPO = Path(tempfile.mkdtemp(prefix='Image_Captioning-hidden-h-'))/'repo'
LOG = Path('/tmp/upt_hidden_h.log')
PROJECT_COMMIT = '31fdec14486ca8ab7ce364a8ffc596f1f9acdd34'
def run(args, cwd=None):
    args = list(map(str, args))
    print('\nRunning:', ' '.join(args), flush=True)
    with LOG.open('a', encoding='utf-8') as log:
        log.write('\nRunning: ' + ' '.join(args) + '\n'); log.flush()
        with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f'Command failed ({status}). Saved log: {LOG}')
def checkout(url, path, revision):
    if not path.exists():
        run(['git','clone',url,path])
    assert (path/'.git').is_dir(), path
    assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=path,text=True).strip(), 'Preserve tracked edits; use a fresh runtime path.'
    run(['git','fetch','--depth','1','origin',revision], cwd=path)
    run(['git','checkout','--detach',revision], cwd=path)
checkout('https://github.com/Supzxjee/Image_Captioning.git', REPO, PROJECT_COMMIT)
SOURCES = json.loads(zlib.decompress(base64.b64decode('...')).decode('utf-8'))
SOURCE_HASHES = {'hidden_data.py': '9285d02c5aa7660a5656654e2efe034941bc24d6f7c300958c06fcca6f0d43c7', 'hidden_model.py': 'a57453d003ae0cf5471690ec6e9299fa10d03b2732cea1d03c6db634bd3fa376', 'extract_hidden.py': '7b1d72bcf9081d3f8854e698e83ebd6fb51b787d59ff225e7b1a00cc89bb1fee', 'run_hidden.py': 'f12b2202e57bcbf4a237ded68844b4d5e551ef231e8906e1e3096abcfe0e3033', 'hidden_metric_smoke.py': '8cf64c56b169cdd2407fdeb668408a80d947578632154405c47c7040558a698b', 'test_hidden_data.py': '0c1d14e26b2f9e7e54a6aae868de5ea5ae960e94d5b7b102728568250bea8bde', 'test_hidden_hook.py': '85de000c32f2f4e5058fb3bf1d98d96d0d1b2c43caca4de11dbbe955dbcb4355', 'test_hidden_model.py': '708a881e4766839db35c6a4c76bca8623e0671dd56418146e9c51a7f48dc7f5e', 'test_hidden_ddp.py': '00ae2983f3fb8128e899a0e97277d892a07f15225d77035a4fb34170ecef7e24', 'test_hidden_hdf5.py': 'ca9180d66eb7d1627763c9da915d81149eb801cd9e316482c601b323c6b602ce', 'excluded_phrase_ids.json': 'b18ab526c32ecb0e687b7b24cfe3b8617bff6998dd0fdeb902d4b66e62bd7475', 'pilot_data.py': '93fa39b1de581d774417840be4d5e0b2b8d52bbf9ee53b479adad2c60a45d3d7', 'pilot_model.py': '879046e943169c30e640fd9b04d6a1edfccd19cd17b5311b104b9ec2dc5bf8f1', 'run_pilot.py': '29b70346cbc6d8505ef94374d111ab827c6acfb3a0a1e106ae104830d23c12c6', 'captioning/upt_reference.py': '22d8172fd5da419a23a6c4f65189e1374c29be2e2f24631f735221896c0a9dac', 'captioning/scst_runtime.py': '3452ccfbd43876dc6381b78d1229d1d6b05fe723e3863a04f31c4a784497f669'}
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == SOURCE_HASHES[name], name
    destination = REPO/name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding='utf-8')
print('Pinned runtime and hash-checked embedded source READY', flush=True)


In [ ]:
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','pycocotools','pycocoevalcap==1.2',
    'h5py','numpy','Pillow','tqdm','nltk==3.9.2','future'])
run([sys.executable,'-m','pip','install','-q','--no-deps',
    'git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'])
run([sys.executable,'-u','-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"])
run(['java','-version'])
CHAIR=Path('/tmp/hidden-caption-chair')
checkout('https://github.com/Maxlinn/CHAIR-metric-standalone.git',CHAIR,'4087a26211aa2339b9a76307cb8f0321ef691d0a')
import torch
assert torch.cuda.device_count()==2,'Select GPU T4x2 for this one arm.'
for name in ('test_hidden_data.py','test_hidden_hdf5.py','test_hidden_model.py'):
    run([sys.executable,'-u',name],cwd=REPO)
run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1',
    '--nproc_per_node=2','--max_restarts=0','test_hidden_ddp.py'],cwd=REPO)
run([sys.executable,'-u','hidden_metric_smoke.py',CHAIR],cwd=REPO)


In [ ]:
CONFIG=Path('/tmp/upt_hidden_config_'+ARM+'.json')
cfg=dict(arm=ARM,checkpoint=str(CHECKPOINT),legacy_prompt=str(LEGACY_PROMPT),visual_cache=str(VISUAL_CACHE),
    dataset_json=str(COCO_JSON),detections=str(DETECTIONS),hidden_cache=str(HIDDEN_CACHE),output=str(OUTPUT),
    stage='/tmp/upt_hidden_stage_'+ARM+'.h5',chair=str(CHAIR),project_commit=PROJECT_COMMIT,
    global_batch_size=8,eval_batch_size=16,warmup_updates=200,base_lr=5e-6,branch_lr=1e-4,scst_weight=.05,seed=42,
    adamw_options=dict(weight_decay=.01,betas=[.9,.999],eps=1e-8,amsgrad=False,maximize=False,
        foreach=None,capturable=False,differentiable=False,fused=None),
    record_optimizer_manifest=True,max_gradient_norm=1.,dropout_policy='model.eval() with gradients enabled',
    implementation_sha256=SOURCE_HASHES,
    locked_numeric_gate=dict(CIDEr_delta_H_minus_A_min=.01,paired_bootstrap95_lower_gt=0.,
        H_CIDEr_gt_original=True,CHAIRs_increase_max=0.,CHAIRi_increase_max=0.,Recall_drop_max=.01),
    promotion_requires='Matched valid A/H, numeric gates and blind human review of supported relations/instances/extra facts; no automatic promotion.')
CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')
PROTOCOL=OUTPUT/'protocol.json'
try:
    run([sys.executable,'-u','run_hidden.py','prepare','--config',CONFIG],cwd=REPO)
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1',
        '--nproc_per_node=2','--max_restarts=0','run_hidden.py','worker','--config',PROTOCOL],cwd=REPO)
    run([sys.executable,'-u','run_hidden.py','evaluate','--config',PROTOCOL],cwd=REPO)
finally:
    if OUTPUT.is_dir():
        shutil.copyfile(LOG,OUTPUT/'notebook.log')
        if PROTOCOL.is_file():
            run([sys.executable,'-u','run_hidden.py','archive','--config',PROTOCOL],cwd=REPO)
        else:
            import zipfile
            with zipfile.ZipFile(str(OUTPUT)+'_review.zip','w',zipfile.ZIP_DEFLATED) as z:
                for path in OUTPUT.rglob('*'):
                    if path.is_file() and path.suffix in ('.json','.log','.py'):
                        z.write(path,path.relative_to(OUTPUT))
print('SEND REVIEW:',str(OUTPUT)+'_review.zip',flush=True)
print('KEEP CHECKPOINT:',OUTPUT/'checkpoint.bin',flush=True)


Gửi ZIP review và giữ riêng checkpoint.bin. Stage HDF5 đặt trong/tmp, không đưa vào dataset checkpoint.
Không dùng điểm của một nhánh để sửa cấu hình nhánh còn lại. Chờ đủ A/H mới so sánh.
Development đã dùng trước; một seed không xác nhận độ ổn định; chưa có kết quả mới hoặc novelty đã chứng minh.
Nếu lỗi trước khi tạo Output, chạy `print(LOG.read_text(encoding='utf-8')[-12000:])` và gửi phần lỗi đầu tiên.